# Generate cycle-level cardiotoxicity modelling table

This notebook expects two sibling directories under `2026_Summer_Research/`:

```text
2026_Summer_Research/
├── MIMIC_IV_dataset_inspection/   ← REPO_ROOT (this notebook lives in cohort_src/)
│   ├── cohort_src/
│   ├── cohort_outputs/
│   ├── tokenization_src/
│   ├── tokenization_outputs/
│   ├── exploration_notebooks/
│   ├── model_src/
│   └── sql_files/
│       ├── cohort_outcomes_sql/
│       ├── diagnoses_sql/
│       ├── drug_cycles_sql/
│       ├── echo_sql/
│       ├── ecg_sql/
│       └── prescriptions_sql/
└── MIMIC_IV_raw_data/             ← DATA_LOCATION (raw data)
    ├── mimic-iv-3.1/
    ├── mimic-iv-ecg/
    └── mimic-iv-echo/
```

The notebook creates one modelling row per patient cycle-like oncology exposure. You can later split in pandas by `subject_id`.


In [7]:
# If needed in your notebook environment:
# %pip install duckdb pandas pyarrow

from pathlib import Path
import os

import duckdb
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 180)

## 1. Configure repository paths

The notebook lives under `cohort_src/`, so `REPO_ROOT` is resolved by walking up
from `__file__` (or the current directory) until a `requirements.txt` sentinel is found.
This works regardless of where Jupyter / VS Code launches the kernel.


In [8]:
from pathlib import Path
import os, duckdb, pandas as pd

def _find_repo_root(start: Path = Path(".").resolve()) -> Path:
    """Walk up from start until a directory containing requirements.txt is found."""
    for p in [start, *start.parents]:
        if (p / "requirements.txt").exists():
            return p
    return start

REPO_ROOT = _find_repo_root()
DATA_LOCATION = Path("/Users/catherinebalajadia/Downloads/2026_Summer_Research/MIMIC_IV_raw_data").resolve()

SQL_ROOT = REPO_ROOT / "sql_files"
DIAGNOSES_SQL_DIR = SQL_ROOT / "diagnoses_sql"
PRESCRIPTIONS_SQL_DIR = SQL_ROOT / "prescriptions_sql"
DRUG_CYCLES_SQL_DIR = SQL_ROOT / "drug_cycles_sql"

MIMIC_HOSP_DIR = DATA_LOCATION / "mimic-iv-3.1/hosp"
MIMIC_ECHO_DIR = DATA_LOCATION / "mimic-iv-echo"

OUTPUT_DIR = REPO_ROOT / "cohort_outputs" / "cycle_modeling_ver2"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("DIAGNOSES_SQL_DIR:", DIAGNOSES_SQL_DIR)
print("PRESCRIPTIONS_SQL_DIR:", PRESCRIPTIONS_SQL_DIR)
print("DRUG_CYCLES_SQL_DIR:", DRUG_CYCLES_SQL_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("MIMIC_HOSP_DIR exists:", MIMIC_HOSP_DIR.exists())
print("MIMIC_ECHO_DIR exists:", MIMIC_ECHO_DIR.exists())


REPO_ROOT: /Users/catherinebalajadia/Downloads/2026_Summer_Research/MIMIC_IV_dataset_inspection
DIAGNOSES_SQL_DIR: /Users/catherinebalajadia/Downloads/2026_Summer_Research/MIMIC_IV_dataset_inspection/exploration_sql_files/diagnoses_sql
PRESCRIPTIONS_SQL_DIR: /Users/catherinebalajadia/Downloads/2026_Summer_Research/MIMIC_IV_dataset_inspection/exploration_sql_files/prescriptions_sql
DRUG_CYCLES_SQL_DIR: /Users/catherinebalajadia/Downloads/2026_Summer_Research/MIMIC_IV_dataset_inspection/exploration_sql_files/drug_cycles_sql
OUTPUT_DIR: /Users/catherinebalajadia/Downloads/2026_Summer_Research/MIMIC_IV_dataset_inspection/data_outputs/cycle_modeling
MIMIC_HOSP_DIR exists: True
MIMIC_ECHO_DIR exists: True


## 2. SQL files expected by this notebook

The base SQL comes from your existing folders. The cycle-labelling SQL lives under:

```text
sql_files/drug_cycles_sql/
```


In [9]:
BASE_SQL_PATHS = [
    DIAGNOSES_SQL_DIR / "active_cancer.sql",
    DIAGNOSES_SQL_DIR / "personal_history_cancer.sql",
    DIAGNOSES_SQL_DIR / "history_and_active.sql",
    PRESCRIPTIONS_SQL_DIR / "prescriptions_count_regex.sql",
]

CYCLE_SQL_PATHS = [
    DRUG_CYCLES_SQL_DIR / "00_parameters_and_windows.sql",
    DRUG_CYCLES_SQL_DIR / "01_drug_classification_and_first_drug.sql",
    DRUG_CYCLES_SQL_DIR / "02_cycle_exposures.sql",
    DRUG_CYCLES_SQL_DIR / "03_lvef_toxicity_events.sql",
    DRUG_CYCLES_SQL_DIR / "04_cv_toxicity_events.sql",
    DRUG_CYCLES_SQL_DIR / "05_first_toxicity_and_observation.sql",
    DRUG_CYCLES_SQL_DIR / "06_final_modeling_table.sql",
]

for path in BASE_SQL_PATHS + CYCLE_SQL_PATHS:
    print("OK" if path.exists() else "MISSING", path)

OK /Users/catherinebalajadia/Downloads/2026_Summer_Research/MIMIC_IV_dataset_inspection/exploration_sql_files/diagnoses_sql/active_cancer.sql
OK /Users/catherinebalajadia/Downloads/2026_Summer_Research/MIMIC_IV_dataset_inspection/exploration_sql_files/diagnoses_sql/personal_history_cancer.sql
OK /Users/catherinebalajadia/Downloads/2026_Summer_Research/MIMIC_IV_dataset_inspection/exploration_sql_files/diagnoses_sql/history_and_active.sql
OK /Users/catherinebalajadia/Downloads/2026_Summer_Research/MIMIC_IV_dataset_inspection/exploration_sql_files/prescriptions_sql/prescriptions_count_regex.sql
OK /Users/catherinebalajadia/Downloads/2026_Summer_Research/MIMIC_IV_dataset_inspection/exploration_sql_files/drug_cycles_sql/00_parameters_and_windows.sql
OK /Users/catherinebalajadia/Downloads/2026_Summer_Research/MIMIC_IV_dataset_inspection/exploration_sql_files/drug_cycles_sql/01_drug_classification_and_first_drug.sql
OK /Users/catherinebalajadia/Downloads/2026_Summer_Research/MIMIC_IV_dataset_

## 3. Helper functions

In [10]:
def execute_sql_file(con: duckdb.DuckDBPyConnection, path: Path) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing SQL file: {path}")

    sql = path.read_text()

    # Make your exploration SQL rerunnable in notebook sessions.
    sql = sql.replace("CREATE VIEW active_cancer", "CREATE OR REPLACE VIEW active_cancer")
    sql = sql.replace("CREATE VIEW oncology_drugs", "CREATE OR REPLACE VIEW oncology_drugs")

    con.execute(sql)


def execute_sql_files(con: duckdb.DuckDBPyConnection, paths: list[Path]) -> None:
    for path in paths:
        print(f"running {path.relative_to(REPO_ROOT) if path.is_relative_to(REPO_ROOT) else path}")
        execute_sql_file(con, path)


def table_exists(con: duckdb.DuckDBPyConnection, name: str) -> bool:
    result = con.execute(
        """
        SELECT COUNT(*) AS n
        FROM information_schema.tables
        WHERE table_name = ?
        """,
        [name],
    ).fetchone()
    return bool(result and result[0] > 0)


def count_rows(con: duckdb.DuckDBPyConnection, name: str) -> int:
    return con.execute(f"SELECT COUNT(*) FROM {name}").fetchone()[0]


def preview(con: duckdb.DuckDBPyConnection, name: str, n: int = 5) -> pd.DataFrame:
    return con.execute(f"SELECT * FROM {name} LIMIT {n}").df()


def describe_view(con: duckdb.DuckDBPyConnection, name: str) -> pd.DataFrame:
    return con.execute(f"DESCRIBE {name}").df()


def write_dataframe(df: pd.DataFrame, output_dir: Path, stem: str) -> None:
    csv_path = output_dir / f"{stem}.csv"
    df.to_csv(csv_path, index=False)
    print(f"wrote {csv_path}")

    parquet_path = output_dir / f"{stem}.parquet"
    try:
        df.to_parquet(parquet_path, index=False)
        print(f"wrote {parquet_path}")
    except Exception as exc:
        print(f"skipped parquet for {stem}: {exc}")

## 4. Connect to DuckDB

The SQL files use relative paths like:

```text
mimic-iv-3.1/hosp/prescriptions.csv
mimic-iv-echo/structured-measurement.csv
```

So the notebook changes the working directory to `DATA_LOCATION` before executing SQL.

In [11]:
os.chdir(DATA_LOCATION)

DB_TARGET = ":memory:"
# Optional persistent DB:
# DB_TARGET = str(REPO_ROOT / "data_outputs" / "cycle_modeling.duckdb")

con = duckdb.connect(DB_TARGET)
print("Connected to", DB_TARGET)
print("Current working directory:", Path.cwd())

Connected to :memory:
Current working directory: /Users/catherinebalajadia/Downloads/2026_Summer_Research/MIMIC_IV_raw_data


## 5. Load base cohort + oncology prescriptions

In [12]:
execute_sql_files(con, BASE_SQL_PATHS)

print("all_cancer_patients exists:", table_exists(con, "all_cancer_patients"))
print("oncology_drugs exists:", table_exists(con, "oncology_drugs"))
print("all_cancer_patients rows:", f"{count_rows(con, 'all_cancer_patients'):,}")
print("oncology_drugs rows:", f"{count_rows(con, 'oncology_drugs'):,}")

running exploration_sql_files/diagnoses_sql/active_cancer.sql
running exploration_sql_files/diagnoses_sql/personal_history_cancer.sql
running exploration_sql_files/diagnoses_sql/history_and_active.sql
running exploration_sql_files/prescriptions_sql/prescriptions_count_regex.sql
all_cancer_patients exists: True
oncology_drugs exists: True
all_cancer_patients rows: 48,477
oncology_drugs rows: 7,743


In [13]:
preview(con, "oncology_drugs", 10)

,subject_id,hadm_id,pharmacy_id,starttime,stoptime,drug
0,10003019,20277210,25458103,2175-11-14 16:00:00,2175-11-15 15:00:00,doxorubicin
1,10003019,22774359,33098165,2175-10-01 17:00:00,2175-10-16 16:00:00,doxorubicin
2,10003400,26090619,59864436,2134-06-06 16:00:00,2134-06-07 19:00:00,lenalidomide (revlimide)15mg
3,10010231,25499227,80302652,2117-11-10 19:00:00,2117-11-13 18:00:00,daunorubicin
4,10012768,20255893,18062841,2111-08-18 15:00:00,2111-08-21 19:00:00,bortezomib
5,10012768,20255893,72049952,2111-08-13 15:00:00,2111-08-18 16:00:00,bortezomib
6,10012768,20255893,94657909,2111-08-13 22:00:00,2111-08-18 14:00:00,bortezomib
7,10016991,27389040,8836777,2124-09-16 12:00:00,2124-09-16 23:00:00,fluorouracil
8,10016991,27389040,12802537,2124-09-16 11:00:00,2124-09-16 23:00:00,fluorouracil
9,10022373,27450651,56873260,2150-05-21 18:00:00,2150-06-04 08:00:00,fluorouracil


## 6. Run cycle-modelling SQL interactively

You can run these cells one by one and inspect each intermediate view.

### 6.1 Parameters and drug-specific windows

In [14]:
execute_sql_file(con, DRUG_CYCLES_SQL_DIR / "00_parameters_and_windows.sql")
preview(con, "drug_toxicity_windows", 20)

,drug_class,toxicity_window_days,window_rationale
0,anthracycline,365,Early-onset CTRCD/HF commonly assessed within 1 year
1,immune_checkpoint_inhibitor,90,"ICI myocarditis usually occurs early, often within weeks to 3 months"
2,her2_targeted,365,HER2-related LV dysfunction commonly monitored over months to 1 year
3,taxane,90,Shorter window for acute/subacute CV events
4,fluoropyrimidine,30,Often acute/subacute ischemia/vasospasm window
5,vegf_inhibitor,180,Hypertension/HF/ischemic risk over months
6,egfr_inhibitor,180,General CV surveillance window
7,tyrosine_kinase_inhibitor,180,General CV surveillance window
8,proteasome_inhibitor,180,HF/ischemia/arrhythmia risk over months
9,immunomodulatory_agent,180,Thrombotic/CV risk over months


### 6.2 Drug classification and first drug anchor

In [15]:
execute_sql_file(con, DRUG_CYCLES_SQL_DIR / "01_drug_classification_and_first_drug.sql")

print("oncology_drugs_classified rows:", f"{count_rows(con, 'oncology_drugs_classified'):,}")
print("cancer_first_drug rows:", f"{count_rows(con, 'cancer_first_drug'):,}")

preview(con, "oncology_drugs_classified", 10)

oncology_drugs_classified rows: 7,743
cancer_first_drug rows: 2,545


,subject_id,hadm_id,pharmacy_id,starttime,stoptime,drug,drug_class
0,10003019,20277210,25458103,2175-11-14 16:00:00,2175-11-15 15:00:00,doxorubicin,anthracycline
1,10003019,22774359,33098165,2175-10-01 17:00:00,2175-10-16 16:00:00,doxorubicin,anthracycline
2,10003400,26090619,59864436,2134-06-06 16:00:00,2134-06-07 19:00:00,lenalidomide (revlimide)15mg,immunomodulatory_agent
3,10010231,25499227,80302652,2117-11-10 19:00:00,2117-11-13 18:00:00,daunorubicin,anthracycline
4,10012768,20255893,18062841,2111-08-18 15:00:00,2111-08-21 19:00:00,bortezomib,proteasome_inhibitor
5,10012768,20255893,72049952,2111-08-13 15:00:00,2111-08-18 16:00:00,bortezomib,proteasome_inhibitor
6,10012768,20255893,94657909,2111-08-13 22:00:00,2111-08-18 14:00:00,bortezomib,proteasome_inhibitor
7,10016991,27389040,8836777,2124-09-16 12:00:00,2124-09-16 23:00:00,fluorouracil,fluoropyrimidine
8,10016991,27389040,12802537,2124-09-16 11:00:00,2124-09-16 23:00:00,fluorouracil,fluoropyrimidine
9,10022373,27450651,56873260,2150-05-21 18:00:00,2150-06-04 08:00:00,fluorouracil,fluoropyrimidine


In [16]:
con.execute("""
SELECT
    drug_class,
    COUNT(*) AS n_rows,
    COUNT(DISTINCT subject_id) AS n_patients
FROM oncology_drugs_classified
GROUP BY drug_class
ORDER BY n_rows DESC
""").df()

,drug_class,n_rows,n_patients
0,anthracycline,3791,1231
1,taxane,1233,502
2,fluoropyrimidine,1067,418
3,proteasome_inhibitor,752,315
4,tyrosine_kinase_inhibitor,395,122
5,immunomodulatory_agent,197,111
6,vegf_inhibitor,101,75
7,her2_targeted,97,50
8,egfr_inhibitor,64,18
9,immune_checkpoint_inhibitor,46,29


### 6.3 Cycle-like exposures

This collapses nearby oncology drug starts into one cycle-like row. Change `cycle_gap_days` in `00_parameters_and_windows.sql` if you want a wider/narrower grouping rule.

In [17]:
execute_sql_file(con, DRUG_CYCLES_SQL_DIR / "02_cycle_exposures.sql")

print("oncology_cycle_exposures rows:", f"{count_rows(con, 'oncology_cycle_exposures'):,}")
preview(con, "oncology_cycle_exposures", 10)

oncology_cycle_exposures rows: 5,420


,subject_id,cycle_number,prediction_time,cycle_start_date,cycle_end_date,n_exposure_start_days_in_cycle,n_prescription_rows_in_cycle,drugs_in_cycle,drug_classes_in_cycle,toxicity_window_days,window_rationales,exposed_anthracycline,exposed_immune_checkpoint_inhibitor,exposed_her2_targeted,exposed_taxane,exposed_fluoropyrimidine,exposed_vegf_inhibitor,exposed_egfr_inhibitor,exposed_tyrosine_kinase_inhibitor,exposed_proteasome_inhibitor,exposed_immunomodulatory_agent
0,10338535,2.0,2138-07-10 14:00:00,2138-07-10,2138-07-10,1,1.0,doxorubicin,anthracycline,365,Early-onset CTRCD/HF commonly assessed within 1 year,1,0,0,0,0,0,0,0,0,0
1,10342737,3.0,2112-07-22 20:00:00,2112-07-22,2112-07-22,2,4.0,*nf* imatinib mesylate,tyrosine_kinase_inhibitor,180,General CV surveillance window,0,0,0,0,0,0,0,1,0,0
2,10533554,4.0,2171-09-21 18:00:00,2171-09-21,2171-09-21,1,1.0,doxorubicin,anthracycline,365,Early-onset CTRCD/HF commonly assessed within 1 year,1,0,0,0,0,0,0,0,0,0
3,10534626,2.0,2183-06-24 20:00:00,2183-06-24,2183-06-24,1,1.0,doxorubicin,anthracycline,365,Early-onset CTRCD/HF commonly assessed within 1 year,1,0,0,0,0,0,0,0,0,0
4,10860673,3.0,2117-11-03 19:00:00,2117-11-03,2117-11-03,1,1.0,doxorubicin,anthracycline,365,Early-onset CTRCD/HF commonly assessed within 1 year,1,0,0,0,0,0,0,0,0,0
5,10860673,6.0,2118-01-05 20:00:00,2118-01-05,2118-01-05,1,1.0,doxorubicin,anthracycline,365,Early-onset CTRCD/HF commonly assessed within 1 year,1,0,0,0,0,0,0,0,0,0
6,10865504,1.0,2178-09-25 20:00:00,2178-09-25,2178-10-02,2,2.0,*nf* imatinib mesylate,tyrosine_kinase_inhibitor,180,General CV surveillance window,0,0,0,0,0,0,0,1,0,0
7,10866278,1.0,2185-08-22 17:00:00,2185-08-22,2185-08-22,1,1.0,bortezomib,proteasome_inhibitor,180,HF/ischemia/arrhythmia risk over months,0,0,0,0,0,0,0,0,1,0
8,10873456,1.0,2133-02-15 09:00:00,2133-02-15,2133-02-15,1,1.0,doxorubicin,anthracycline,365,Early-onset CTRCD/HF commonly assessed within 1 year,1,0,0,0,0,0,0,0,0,0
9,10877494,1.0,2131-05-04 13:00:00,2131-05-04,2131-05-04,1,1.0,doxorubicin,anthracycline,365,Early-onset CTRCD/HF commonly assessed within 1 year,1,0,0,0,0,0,0,0,0,0


In [18]:
con.execute("""
SELECT
    cycle_number,
    COUNT(*) AS n_rows,
    COUNT(DISTINCT subject_id) AS n_patients
FROM oncology_cycle_exposures
GROUP BY cycle_number
ORDER BY cycle_number
LIMIT 25
""").df()

,cycle_number,n_rows,n_patients
0,1.0,2545,2545
1,2.0,1057,1057
2,3.0,621,621
3,4.0,472,472
4,5.0,347,347
5,6.0,261,261
6,7.0,47,47
7,8.0,30,30
8,9.0,17,17
9,10.0,6,6


### 6.4 LVEF toxicity events

In [19]:
execute_sql_file(con, DRUG_CYCLES_SQL_DIR / "03_lvef_toxicity_events.sql")

print("all_lvef rows:", f"{count_rows(con, 'all_lvef'):,}")
print("baseline_lvef_pre_first_drug rows:", f"{count_rows(con, 'baseline_lvef_pre_first_drug'):,}")
print("lvef_toxicity_events rows:", f"{count_rows(con, 'lvef_toxicity_events'):,}")

preview(con, "lvef_toxicity_events", 10)

all_lvef rows: 147,794
baseline_lvef_pre_first_drug rows: 2,545
lvef_toxicity_events rows: 338


,subject_id,toxicity_time,toxicity_type,baseline_time,baseline_lvef,event_lvef,absolute_lvef_drop
0,10623751,2182-07-07 10:45:00,lvef_ctrctd,2181-03-13 11:55:00,70.0,40.0,30.0
1,10867608,2185-01-07 17:00:00,lvef_ctrctd,2184-11-05 14:43:00,65.0,45.0,20.0
2,11111901,2164-04-08 15:44:00,lvef_ctrctd,2163-08-24 09:53:00,55.0,30.0,25.0
3,11564416,2154-07-18 11:42:00,lvef_ctrctd,2154-05-04 12:31:00,55.0,45.0,10.0
4,11567253,2177-10-16 10:53:00,lvef_ctrctd,2177-10-05 09:30:00,70.0,35.0,35.0
5,11567253,2177-10-25 09:49:00,lvef_ctrctd,2177-10-05 09:30:00,70.0,35.0,35.0
6,11567778,2171-12-22 14:56:00,lvef_ctrctd,2171-10-10 15:51:00,60.0,40.0,20.0
7,11567778,2172-01-22 11:00:00,lvef_ctrctd,2171-10-10 15:51:00,60.0,41.0,19.0
8,11567778,2172-04-05 10:00:00,lvef_ctrctd,2171-10-10 15:51:00,60.0,45.0,15.0
9,11567778,2172-04-19 16:09:00,lvef_ctrctd,2171-10-10 15:51:00,60.0,45.0,15.0


### 6.5 ICD/admission cardiovascular toxicity events

In [20]:
execute_sql_file(con, DRUG_CYCLES_SQL_DIR / "04_cv_toxicity_events.sql")

print("cancer_cv_admission_events rows:", f"{count_rows(con, 'cancer_cv_admission_events'):,}")
print("cv_toxicity_events rows:", f"{count_rows(con, 'cv_toxicity_events'):,}")
print("pre_existing_cv_history rows:", f"{count_rows(con, 'pre_existing_cv_history'):,}")

preview(con, "cv_toxicity_events", 10)

cancer_cv_admission_events rows: 17,962
cv_toxicity_events rows: 2,827
pre_existing_cv_history rows: 813


,subject_id,toxicity_time,toxicity_type,hadm_id,cv_event_icd_codes
0,13207656,2179-04-11 01:05:00,cv_diagnosis_admission,26583023,1970 | 1977 | 1985 | 1987 | 1991 | 2761 | 3051 | 3383 | 4019 | 41071 | 42789 | 42989 | 49390 | 78702 | E9352 | V153 ...
1,15501251,2154-04-11 21:27:00,cv_diagnosis_admission,29635808,C9000 | D630 | D6959 | E43 | F17210 | F329 | F419 | G893 | I130 | I480 | I5043 | I5181 | I959 | J9811 | L89891 | N18...
2,13154986,2134-10-05 10:32:00,cv_diagnosis_admission,25017229,20400 | 25541 | 27652 | 27951 | 28803 | 311 | 3576 | 4280 | 4580 | 486 | 52801 | 5589 | 6924 | 78061 | 7821 | 79001 ...
3,13162333,2138-10-03 17:31:00,cv_diagnosis_admission,20653522,C9002 | D591 | D6481 | D696 | D72819 | E1121 | E1122 | E11319 | E1140 | G3184 | I130 | I5032 | N183 | R55 | T451X5A ...
4,13162333,2139-04-16 18:47:00,cv_diagnosis_admission,24806417,C9002 | D589 | E1121 | E1122 | E11319 | E1140 | I130 | I5032 | N179 | N183 | T375X1A | T461X1A | T501X1A | T504X1A |...
5,13375185,2186-05-06 21:38:00,cv_diagnosis_admission,23687981,B450 | C8338 | E1122 | E1151 | E43 | E860 | E872 | E875 | F05 | G40501 | G4733 | G9340 | I130 | I2510 | I252 | I255 ...
6,13386704,2143-06-06 20:02:00,cv_diagnosis_admission,25887116,20190 | 28803 | 42789 | 4580 | 52801 | 53081 | 6071 | 78061 | 7840 | 79319 | E9331
7,13386704,2143-04-22 16:29:00,cv_diagnosis_admission,29516967,20190 | 42789 | 53081 | 78039 | 7840
8,13425067,2135-02-08 18:49:00,cv_diagnosis_admission,21323682,C9150 | D649 | D65 | E46 | E8342 | E876 | I471 | N2589 | Z6823
9,13425424,2118-02-05 00:56:00,cv_diagnosis_admission,28390570,A4102 | A4189 | C9000 | D61818 | F17210 | G4733 | G629 | I4891 | I4892 | I5032 | K521 | M109 | M4624 | T451X5A | T80...


### 6.6 First toxicity and observation evidence

In [21]:
execute_sql_file(con, DRUG_CYCLES_SQL_DIR / "05_first_toxicity_and_observation.sql")

print("all_cardiotoxicity_events rows:", f"{count_rows(con, 'all_cardiotoxicity_events'):,}")
print("first_cardiotoxicity_event rows:", f"{count_rows(con, 'first_cardiotoxicity_event'):,}")
print("patient_last_observation rows:", f"{count_rows(con, 'patient_last_observation'):,}")

preview(con, "first_cardiotoxicity_event", 10)

all_cardiotoxicity_events rows: 3,165
first_cardiotoxicity_event rows: 867
patient_last_observation rows: 229,103


,subject_id,first_toxicity_time,first_toxicity_type,first_event_lvef,first_absolute_lvef_drop,first_cv_event_icd_codes
0,10960463,2156-10-20 00:00:00,cv_diagnosis_admission,NaN,NaN,C569 | C785 | C786 | E785 | I10 | I471 | I7300 | J986 | K7689 | M069 | N135 | N323 | N736 | R339 | T8110XA | Y838 | ...
1,13683698,2178-01-02 19:01:00,cv_diagnosis_admission,NaN,NaN,1309 | 1961 | 1970 | 1983 | 27542 | 27651 | 30503 | 4019 | 4271 | 5070 | 5180 | 51884 | 5362 | 78321 | 78559 | 99594...
2,19459111,2114-02-18 17:48:00,cv_diagnosis_admission,NaN,NaN,B370 | B957 | C8332 | C9200 | D709 | F419 | I340 | I4892 | I951 | J910 | K219 | L89152 | N390 | R110 | R51 | R627 | ...
3,11408373,2193-01-08 04:13:00,cv_diagnosis_admission,NaN,NaN,0088 | 2352 | 3051 | 42731 | 53081 | 56089 | 60000
4,11575755,2134-11-01 19:05:00,cv_diagnosis_admission,NaN,NaN,C9002 | D539 | D709 | E1122 | E785 | G620 | I130 | I340 | I4820 | I5022 | K219 | K529 | N179 | N189 | N390 | T451X5A...
5,12500913,2166-11-20 15:54:00,cv_diagnosis_admission,NaN,NaN,B349 | C9202 | D61818 | E041 | E8339 | I110 | I5022 | I959 | T451X5S | Y842 | Z006 | Z20822 | Z66 | Z853
6,14098347,2149-10-05 15:26:00,cv_diagnosis_admission,NaN,NaN,C9000 | E669 | F329 | G4700 | G629 | I10 | I214 | I221 | I25119 | I252 | M545 | Z7902 | Z86718 | Z9484
7,15584013,2131-06-11 05:46:00,cv_diagnosis_admission,NaN,NaN,C9202 | D709 | D89811 | E0500 | E876 | F329 | F419 | G8929 | H16209 | I5023 | I629 | J156 | J45909 | K219 | Q211 | R...
8,19519825,2172-03-22 19:27:00,cv_diagnosis_admission,NaN,NaN,A047 | B370 | C8339 | D510 | D709 | D72829 | E039 | E222 | E46 | E8809 | I4891 | I82B11 | I872 | I898 | K1231 | K219...
9,11841078,2175-11-23 07:41:00,cv_diagnosis_admission,NaN,NaN,C6290 | C7800 | C787 | C7889 | C7931 | C7932 | C7951 | D62 | D6481 | E099 | E872 | G893 | I472 | J90 | J942 | J9621 ...


### 6.7 Final modelling table

In [22]:
execute_sql_file(con, DRUG_CYCLES_SQL_DIR / "06_final_modeling_table.sql")

print("final_cycle_modeling_table rows:", f"{count_rows(con, 'final_cycle_modeling_table'):,}")
print("final_cycle_binary_modeling_table rows:", f"{count_rows(con, 'final_cycle_binary_modeling_table'):,}")

preview(con, "final_cycle_modeling_table", 10)

final_cycle_modeling_table rows: 5,420
final_cycle_binary_modeling_table rows: 2,555


,subject_id,cycle_number,prediction_time,cycle_start_date,cycle_end_date,n_exposure_start_days_in_cycle,n_prescription_rows_in_cycle,drugs_in_cycle,drug_classes_in_cycle,toxicity_window_days,prediction_window_end,window_rationales,exposed_anthracycline,exposed_immune_checkpoint_inhibitor,exposed_her2_targeted,exposed_taxane,exposed_fluoropyrimidine,exposed_vegf_inhibitor,exposed_egfr_inhibitor,exposed_tyrosine_kinase_inhibitor,exposed_proteasome_inhibitor,exposed_immunomodulatory_agent,first_toxicity_time,first_toxicity_type,first_event_lvef,first_absolute_lvef_drop,first_cv_event_icd_codes,toxicity_in_window,pre_existing_cv_history,baseline_time,baseline_lvef,last_observation_time,observed_through_prediction_window,has_observation_in_prediction_window,eligible_for_binary_label,binary_label,label
0,11408373,2.0,2193-01-08 21:00:00,2193-01-08,2193-01-08,1,1.0,imatinib mesylate,tyrosine_kinase_inhibitor,180,2193-07-07 21:00:00,General CV surveillance window,0,0,0,0,0,0,0,1,0,0,2193-01-08 04:13:00,cv_diagnosis_admission,NaN,NaN,0088 | 2352 | 3051 | 42731 | 53081 | 56089 | 60000,0,1,NaT,NaN,2198-05-14 00:00:00,1,1,0,<NA>,exclude_already_toxic
1,11575755,2.0,2134-11-03 11:00:00,2134-11-03,2134-11-03,1,1.0,bortezomib,proteasome_inhibitor,180,2135-05-02 11:00:00,HF/ischemia/arrhythmia risk over months,0,0,0,0,0,0,0,0,1,0,2134-11-01 19:05:00,cv_diagnosis_admission,NaN,NaN,C9002 | D539 | D709 | E1122 | E785 | G620 | I130 | I340 | I4820 | I5022 | K219 | K529 | N179 | N189 | N390 | T451X5A...,0,1,NaT,NaN,2134-11-03 11:00:00,0,0,0,<NA>,exclude_already_toxic
2,19519825,5.0,2172-05-02 14:00:00,2172-05-02,2172-05-02,2,4.0,doxorubicin,anthracycline,365,2173-05-02 14:00:00,Early-onset CTRCD/HF commonly assessed within 1 year,1,0,0,0,0,0,0,0,0,0,2172-03-22 19:27:00,cv_diagnosis_admission,NaN,NaN,A047 | B370 | C8339 | D510 | D709 | D72829 | E039 | E222 | E46 | E8809 | I4891 | I82B11 | I872 | I898 | K1231 | K219...,0,1,2172-01-15 14:18:00,55.0,2172-06-11 10:00:00,0,1,0,<NA>,exclude_already_toxic
3,12594999,1.0,2178-07-01 12:00:00,2178-07-01,2178-07-01,1,1.0,doxorubicin,anthracycline,365,2179-07-01 12:00:00,Early-onset CTRCD/HF commonly assessed within 1 year,1,0,0,0,0,0,0,0,0,0,2178-07-21 03:35:00,cv_diagnosis_admission,NaN,NaN,C8330 | E119 | I471 | K7469 | K7581 | R188 | R402142 | R402252 | R402362 | S02119A | S066X0A | W108XXA | Y92038 | Z2...,1,1,2178-06-09 12:04:00,75.0,2178-09-21 18:24:00,0,1,1,1,positive
4,10588464,3.0,2113-04-06 16:00:00,2113-04-06,2113-04-11,2,2.0,doxorubicin,anthracycline,365,2114-04-06 16:00:00,Early-onset CTRCD/HF commonly assessed within 1 year,1,0,0,0,0,0,0,0,0,0,2113-03-16 00:00:00,cv_diagnosis_admission,NaN,NaN,C8339 | E039 | E279 | E785 | E8770 | I10 | I480 | J158 | J449 | K5900 | R739 | T380X5A | Y92239 | Z5111 | Z7902 | Z8...,0,1,2113-02-20 13:54:00,55.0,2115-03-24 13:00:00,1,1,0,<NA>,exclude_already_toxic
5,14493463,2.0,2126-07-19 12:00:00,2126-07-19,2126-07-19,1,1.0,liposomal doxorubicin (doxil),anthracycline,365,2127-07-19 12:00:00,Early-onset CTRCD/HF commonly assessed within 1 year,1,0,0,0,0,0,0,0,0,0,2126-03-25 20:00:00,lvef_ctrctd,30.0,30.0,NaN,0,1,2126-03-22 13:06:00,60.0,2127-06-28 16:08:00,0,1,0,<NA>,exclude_already_toxic
6,17587395,1.0,2129-04-10 13:00:00,2129-04-10,2129-04-10,1,1.0,inv-lenalidomide,immunomodulatory_agent,180,2129-10-07 13:00:00,Thrombotic/CV risk over months,0,0,0,0,0,0,0,0,0,1,2129-05-21 15:05:00,cv_diagnosis_admission,NaN,NaN,0059 | 00845 | 0088 | 20502 | 25000 | 3688 | 4019 | 41401 | 4387 | 4871 | 5651 | V1582 | V168 | V173 | V4582 | V8741,1,1,2129-04-03 11:00:00,55.0,2130-05-13 17:41:00,1,1,1,1,positive
7,19521943,2.0,2131-10-09 14:00:00,2131-10-09,2131-10-09,2,4.0,liposomal doxorubicin (doxil) | paclitaxel (taxol),anthracycline | taxane,365,2132-10-08 14:00:00,Early-onset CTRCD/HF commonly assessed within 1 year | Shorter window for acute/subacute CV events,1,0,0,1,0,0,0,0,0,0,2131-10-13 16:57:00,cv_diagnosis_admission,NaN,NaN,A0472 | B952 | B965 | C50411 | C50911 | C50912 | C77

## 7. Load final views into pandas

In [23]:
final_df = con.execute("""
SELECT *
FROM final_cycle_modeling_table
ORDER BY subject_id, cycle_number
""").df()

binary_df = con.execute("""
SELECT *
FROM final_cycle_binary_modeling_table
ORDER BY subject_id, cycle_number
""").df()

final_df.shape, binary_df.shape

((5420, 37), (2555, 37))

In [24]:
final_df.head(10)

,subject_id,cycle_number,prediction_time,cycle_start_date,cycle_end_date,n_exposure_start_days_in_cycle,n_prescription_rows_in_cycle,drugs_in_cycle,drug_classes_in_cycle,toxicity_window_days,prediction_window_end,window_rationales,exposed_anthracycline,exposed_immune_checkpoint_inhibitor,exposed_her2_targeted,exposed_taxane,exposed_fluoropyrimidine,exposed_vegf_inhibitor,exposed_egfr_inhibitor,exposed_tyrosine_kinase_inhibitor,exposed_proteasome_inhibitor,exposed_immunomodulatory_agent,first_toxicity_time,first_toxicity_type,first_event_lvef,first_absolute_lvef_drop,first_cv_event_icd_codes,toxicity_in_window,pre_existing_cv_history,baseline_time,baseline_lvef,last_observation_time,observed_through_prediction_window,has_observation_in_prediction_window,eligible_for_binary_label,binary_label,label
0,10003019,1.0,2175-10-01 17:00:00,2175-10-01,2175-10-01,1,1.0,doxorubicin,anthracycline,365,2176-09-30 17:00:00,Early-onset CTRCD/HF commonly assessed within 1 year,1,0,0,0,0,0,0,0,0,0,2175-10-08 13:56:00,cv_diagnosis_admission,NaN,NaN,0389 | 135 | 20190 | 2724 | 2753 | 2859 | 311 | 32723 | 36411 | 4270 | 486 | 5178 | 78061 | 78552 | 78791 | 79092 | ...,1,1,2175-09-25 17:00:00,60.0,2184-03-14 13:57:00,1,1,1,1,positive
1,10003019,2.0,2175-11-14 16:00:00,2175-11-14,2175-11-14,1,1.0,doxorubicin,anthracycline,365,2176-11-13 16:00:00,Early-onset CTRCD/HF commonly assessed within 1 year,1,0,0,0,0,0,0,0,0,0,2175-10-08 13:56:00,cv_diagnosis_admission,NaN,NaN,0389 | 135 | 20190 | 2724 | 2753 | 2859 | 311 | 32723 | 36411 | 4270 | 486 | 5178 | 78061 | 78552 | 78791 | 79092 | ...,0,1,2175-09-25 17:00:00,60.0,2184-03-14 13:57:00,1,1,0,<NA>,exclude_already_toxic
2,10003400,1.0,2134-06-06 16:00:00,2134-06-06,2134-06-06,1,1.0,lenalidomide (revlimide)15mg,immunomodulatory_agent,180,2134-12-03 16:00:00,Thrombotic/CV risk over months,0,0,0,0,0,0,0,0,0,1,2136-11-04 20:43:00,cv_diagnosis_admission,NaN,NaN,1548 | 20300 | 27800 | 28412 | 2851 | 40390 | 42731 | 4589 | 56210 | 56400 | 5853 | 71596 | 7919 | E8490 | E9478 | V...,0,1,2134-04-04 14:00:00,55.0,2137-08-05 14:31:00,1,0,1,0,negative_observed
3,10010231,1.0,2117-11-10 19:00:00,2117-11-10,2117-11-10,1,1.0,daunorubicin,anthracycline,365,2118-11-10 19:00:00,Early-onset CTRCD/HF commonly assessed within 1 year,1,0,0,0,0,0,0,0,0,0,NaT,NaN,NaN,NaN,NaN,0,0,2117-11-09 09:00:00,60.0,2118-04-17 11:03:00,0,1,0,<NA>,unknown_no_followup_evidence
4,10012768,1.0,2111-08-13 15:00:00,2111-08-13,2111-08-18,3,5.0,bortezomib,proteasome_inhibitor,180,2112-02-09 15:00:00,HF/ischemia/arrhythmia risk over months,0,0,0,0,0,0,0,0,1,0,NaT,NaN,NaN,NaN,NaN,0,0,NaT,NaN,2112-01-06 15:01:00,0,1,0,<NA>,unknown_no_followup_evidence
5,10016991,1.0,2124-09-16 11:00:00,2124-09-16,2124-09-16,2,4.0,fluorouracil,fluoropyrimidine,30,2124-10-16 11:00:00,Often acute/subacute ischemia/vasospasm window,0,0,0,0,1,0,0,0,0,0,NaT,NaN,NaN,NaN,NaN,0,0,NaT,NaN,2124-09-16 12:00:00,0,1,0,<NA>,unknown_no_followup_evidence
6,10022373,1.0,2150-05-21 18:00:00,2150-05-21,2150-05-21,2,4.0,fluorouracil,fluoropyrimidine,30,2150-06-20 18:00:00,Often acute/subacute ischemia/vasospasm window,0,0,0,0,1,0,0,0,0,0,NaT,NaN,NaN,NaN,NaN,0,0,NaT,NaN,2150-07-19 00:26:00,1,0,1,0,negative_observed
7,10027393,1.0,2128-10-30 18:00:00,2128-10-30,2128-10-30,2,4.0,fluorouracil,fluoropyrimidine,30,2128-11-29 18:00:00,Often acute/subacute ischemia/vasospasm window,0,0,0,0,1,0,0,0,0,0,NaT,NaN,NaN,NaN,NaN,0,0,NaT,NaN,2128-10-30 18:00:00,0,0,0,<NA>,unknown_no_followup_evidence
8,10030549,1.0,2141-08-16 10:00:00,2141-08-16,2141-08-16,2,4.0,paclitaxel (taxol),taxane,90,2141-11-14 10:00:00,Shorter window for acute/subacute CV events,0,0,0,1,0,0,0,0,0,0,NaT,NaN,NaN,NaN,NaN,0,0,NaT,NaN,2141-11-12 01:45:00,0,1,0,<NA>,unknown_no_followup_evidence
9,10030549,2.0,2141-09-05 14:00:00,2141-09-05,2141-09-05,1,1.0,paclitaxel (taxol),taxane,90,2141-12-04 14:00:00,Shorter window for acute/subacute CV events,0,0,0,1,0,0,0,0,0,0,NaT,NaN,NaN,NaN,NaN,0,0,NaT,NaN,2141-11-12 01:45:00,0,1,0,<NA>,unknown_n

## 8. Explore label quality

In [25]:
label_breakdown = (
    final_df.groupby("label", dropna=False)
    .agg(
        n_cycle_rows=("subject_id", "size"),
        n_patients_with_at_least_one_row=("subject_id", "nunique"),
    )
    .reset_index()
    .sort_values("n_cycle_rows", ascending=False)
)

label_breakdown

,label,n_cycle_rows,n_patients_with_at_least_one_row
3,unknown_no_followup_evidence,2139,1194
1,negative_observed,1604,860
2,positive,951,644
0,exclude_already_toxic,726,292


In [26]:
binary_label_breakdown = (
    binary_df.groupby(["label", "binary_label"], dropna=False)
    .agg(
        n_cycle_rows=("subject_id", "size"),
        n_patients_with_at_least_one_row=("subject_id", "nunique"),
    )
    .reset_index()
    .sort_values("n_cycle_rows", ascending=False)
)

binary_label_breakdown

,label,binary_label,n_cycle_rows,n_patients_with_at_least_one_row
0,negative_observed,0,1604,860
1,positive,1,951,644


In [27]:
drug_class_breakdown = (
    final_df.groupby("drug_classes_in_cycle", dropna=False)
    .agg(
        n_cycle_rows=("subject_id", "size"),
        n_patients_with_at_least_one_row=("subject_id", "nunique"),
        n_positive_cycle_rows=("toxicity_in_window", "sum"),
    )
    .reset_index()
    .sort_values("n_cycle_rows", ascending=False)
)

drug_class_breakdown.head(25)

,drug_classes_in_cycle,n_cycle_rows,n_patients_with_at_least_one_row,n_positive_cycle_rows
0,anthracycline,3003,1198,621
25,taxane,674,449,81
14,fluoropyrimidine,559,368,39
24,proteasome_inhibitor,441,275,94
27,tyrosine_kinase_inhibitor,201,117,39
22,immunomodulatory_agent,103,83,19
5,anthracycline | proteasome_inhibitor,84,31,6
28,vegf_inhibitor,80,65,3
16,fluoropyrimidine | taxane,52,35,11
18,her2_targeted,46,31,7


In [28]:
def assign_patient_status(labels):
    labels = set(labels)

    if "positive" in labels:
        return "positive_patient"
    elif "negative_observed" in labels:
        return "negative_observed_patient"
    elif "unknown_no_followup_evidence" in labels:
        return "unknown_patient"
    elif "exclude_already_toxic" in labels:
        return "only_excluded_rows_review"
    else:
        return "unclassified_review"


patient_level_labels = (
    final_df
    .groupby("subject_id")["label"]
    .apply(assign_patient_status)
    .reset_index(name="patient_status")
)

patient_level_summary = (
    patient_level_labels
    .groupby("patient_status")
    .agg(n_patients=("subject_id", "nunique"))
    .reset_index()
    .sort_values("n_patients", ascending=False)
)

patient_level_summary

,patient_status,n_patients
2,unknown_patient,1076
0,negative_observed_patient,825
1,positive_patient,644


In [29]:
cohort_accounting = pd.DataFrame({
    "metric": [
        "patients_in_cancer_first_drug",
        "patients_in_final_cycle_modeling_table",
        "patients_in_binary_modeling_table",
    ],
    "n_patients": [
        con.execute("SELECT COUNT(DISTINCT subject_id) FROM cancer_first_drug").fetchone()[0],
        final_df["subject_id"].nunique(),
        binary_df["subject_id"].nunique(),
    ],
})

cohort_accounting

,metric,n_patients
0,patients_in_cancer_first_drug,2545
1,patients_in_final_cycle_modeling_table,2545
2,patients_in_binary_modeling_table,1469


In [30]:
# Good sanity check: inspect patients with many cycle rows.
example_subjects = (
    final_df.groupby("subject_id")
    .size()
    .sort_values(ascending=False)
    .head(10)
    .index
)

final_df[final_df["subject_id"].isin(example_subjects)].sort_values(["subject_id", "cycle_number"])

,subject_id,cycle_number,prediction_time,cycle_start_date,cycle_end_date,n_exposure_start_days_in_cycle,n_prescription_rows_in_cycle,drugs_in_cycle,drug_classes_in_cycle,toxicity_window_days,prediction_window_end,window_rationales,exposed_anthracycline,exposed_immune_checkpoint_inhibitor,exposed_her2_targeted,exposed_taxane,exposed_fluoropyrimidine,exposed_vegf_inhibitor,exposed_egfr_inhibitor,exposed_tyrosine_kinase_inhibitor,exposed_proteasome_inhibitor,exposed_immunomodulatory_agent,first_toxicity_time,first_toxicity_type,first_event_lvef,first_absolute_lvef_drop,first_cv_event_icd_codes,toxicity_in_window,pre_existing_cv_history,baseline_time,baseline_lvef,last_observation_time,observed_through_prediction_window,has_observation_in_prediction_window,eligible_for_binary_label,binary_label,label
557,11023442,1.0,2174-12-25 12:00:00,2174-12-25,2174-12-25,3,9.0,liposomal doxorubicin (doxil),anthracycline,365,2175-12-25 12:00:00,Early-onset CTRCD/HF commonly assessed within 1 year,1,0,0,0,0,0,0,0,0,0,NaT,NaN,NaN,NaN,NaN,0,0,NaT,NaN,2176-03-09 08:00:00,1,1,1,0,negative_observed
558,11023442,2.0,2175-01-22 10:00:00,2175-01-22,2175-01-22,3,9.0,liposomal doxorubicin (doxil),anthracycline,365,2176-01-22 10:00:00,Early-onset CTRCD/HF commonly assessed within 1 year,1,0,0,0,0,0,0,0,0,0,NaT,NaN,NaN,NaN,NaN,0,0,NaT,NaN,2176-03-09 08:00:00,1,1,1,0,negative_observed
559,11023442,3.0,2175-02-26 10:00:00,2175-02-26,2175-02-26,3,9.0,liposomal doxorubicin (doxil),anthracycline,365,2176-02-26 10:00:00,Early-onset CTRCD/HF commonly assessed within 1 year,1,0,0,0,0,0,0,0,0,0,NaT,NaN,NaN,NaN,NaN,0,0,NaT,NaN,2176-03-09 08:00:00,1,1,1,0,negative_observed
560,11023442,4.0,2175-03-26 10:00:00,2175-03-26,2175-03-26,3,9.0,liposomal doxorubicin (doxil),anthracycline,365,2176-03-25 10:00:00,Early-onset CTRCD/HF commonly assessed within 1 year,1,0,0,0,0,0,0,0,0,0,NaT,NaN,NaN,NaN,NaN,0,0,NaT,NaN,2176-03-09 08:00:00,0,1,0,<NA>,unknown_no_followup_evidence
561,11023442,5.0,2175-04-23 09:00:00,2175-04-23,2175-04-23,3,9.0,liposomal doxorubicin (doxil),anthracycline,365,2176-04-22 09:00:00,Early-onset CTRCD/HF commonly assessed within 1 year,1,0,0,0,0,0,0,0,0,0,NaT,NaN,NaN,NaN,NaN,0,0,NaT,NaN,2176-03-09 08:00:00,0,1,0,<NA>,unknown_no_followup_evidence
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4034,17424221,9.0,2167-03-17 14:00:00,2167-03-17,2167-03-17,1,1.0,doxorubicin,anthracycline,365,2168-03-16 14:00:00,Early-onset CTRCD/HF commonly assessed within 1 year,1,0,0,0,0,0,0,0,0,0,2166-08-21 15:00:00,lvef_ctrctd,45.0,15.0,NaN,0,0,2166-04-14 13:44:00,60.0,2167-08-19 18:00:00,0,1,0,<NA>,exclude_already_toxic
4035,17424221,10.0,2167-05-05 08:00:00,2167-05-05,2167-05-08,3,5.0,bortezomib | doxorubicin | pomalidomide,anthracycline | immunomodulatory_agent | proteasome_inhibitor,365,2168-05-04 08:00:00,Early-onset CTRCD/HF commonly assessed within 1 year | HF/ischemia/arrhythmia risk over months | Thrombotic/CV risk ...,1,0,0,0,0,0,0,0,1,1,2166-08-21 15:00:00,lvef_ctrctd,45.0,15.0,NaN,0,0,2166-04-14 13:44:00,60.0,2167-08-19 18:00:00,0,1,0,<NA>,exclude_already_toxic
4036,17424221,11.0,2167-06-02 17:00:00,2167-06-02,2167-06-02,2,4.0,doxorubicin | pomalidomide,anthracycline | immunomodulatory_agent,365,2168-06-01 17:00:00,Early-onset CTRCD/HF commonly assessed within 1 year | Thrombotic/CV risk over months,1,0,0,0,0,0,0,0,0,1,2166-08-21 15:00:00,lvef_ctrctd,45.0,15.0,NaN,0,0,2166-04-14 13:44:00,60.0,2167-08-19 18:00:00,0,1,0,<NA>,exclude_already_toxic
4037,17424221,12.0,2167-06-28 14:00:00,2167-06-28,2167-06-29,3,5.0,doxorubicin | pomalidomide,anthracycline | immunomodulatory_agent,365,2168-06-27 14:00:00,Early-onset CTRCD/HF commonly assessed within 1 year | Thrombotic/CV risk over months,1,0,0,0,0,0,0,0,0,1,2166-08-21 15:00:00,lvef_ctrctd,45.0,15.0,NaN,0,0,2166-04-14 13:44:00,60.0,2167-08-19 18:00:00,0,1,0,<NA>,exclude_already_toxic


## 9. Export modelling tables

In [31]:
write_dataframe(final_df, OUTPUT_DIR, "final_cycle_modeling_table")
write_dataframe(binary_df, OUTPUT_DIR, "final_cycle_binary_modeling_table")

label_breakdown.to_csv(OUTPUT_DIR / "row_level_label_breakdown.csv", index=False)
binary_label_breakdown.to_csv(OUTPUT_DIR / "row_level_binary_label_breakdown.csv", index=False)
drug_class_breakdown.to_csv(OUTPUT_DIR / "row_level_drug_class_breakdown.csv", index=False)

patient_level_labels.to_csv(OUTPUT_DIR / "patient_level_labels.csv", index=False)
patient_level_summary.to_csv(OUTPUT_DIR / "patient_level_summary.csv", index=False)
cohort_accounting.to_csv(OUTPUT_DIR / "cohort_accounting.csv", index=False)
print("Wrote outputs to", OUTPUT_DIR)

wrote /Users/catherinebalajadia/Downloads/2026_Summer_Research/MIMIC_IV_dataset_inspection/data_outputs/cycle_modeling/final_cycle_modeling_table.csv
wrote /Users/catherinebalajadia/Downloads/2026_Summer_Research/MIMIC_IV_dataset_inspection/data_outputs/cycle_modeling/final_cycle_modeling_table.parquet
wrote /Users/catherinebalajadia/Downloads/2026_Summer_Research/MIMIC_IV_dataset_inspection/data_outputs/cycle_modeling/final_cycle_binary_modeling_table.csv
wrote /Users/catherinebalajadia/Downloads/2026_Summer_Research/MIMIC_IV_dataset_inspection/data_outputs/cycle_modeling/final_cycle_binary_modeling_table.parquet
Wrote outputs to /Users/catherinebalajadia/Downloads/2026_Summer_Research/MIMIC_IV_dataset_inspection/data_outputs/cycle_modeling


## 10. Optional patient-level split template

Only use this after you are ready to model. The split must be by `subject_id`, not by row/cycle.

In [32]:
# Optional template:
# from sklearn.model_selection import train_test_split
#
# patient_ids = binary_df["subject_id"].drop_duplicates()
# train_ids, test_ids = train_test_split(patient_ids, test_size=0.2, random_state=42)
#
# train_df = binary_df[binary_df["subject_id"].isin(train_ids)].copy()
# test_df = binary_df[binary_df["subject_id"].isin(test_ids)].copy()
#
# print(train_df.shape, test_df.shape)
# print("Patient overlap:", set(train_df.subject_id) & set(test_df.subject_id))